# ESC-Fault v2 starter: multi-fault diagnosis of electrostatic chucks

Simulated sensor data from 552 electrostatic chucks (the part of an etch tool that clamps and cools the wafer).
v2 adds an `arcing` fault, chucks with **two faults at once** (so diagnosis is multi-label), intermittent leaks,
slow onsets, a recipe that appears mid-life, and a `test_shift` split from a second tool.

This notebook builds causal features, trains one gradient-boosted classifier per fault, scores it on the
validation split, breaks down the hardest fault (`sensor_drift`), and writes test prediction files.

- Code, simulator, scoring: https://github.com/forgottenl5/esc-fault-benchmark
- Data: https://huggingface.co/datasets/forgottenl5/esc-fault-benchmark (folder `v2/`)
- Cite: https://doi.org/10.5281/zenodo.24185993

The data is **simulated**; good scores do not prove a method works on a real tool. Runtime: about 5 minutes on a laptop.

In [1]:
import glob, json
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

def find(name):
    pats = [f"/kaggle/input/**/v2/**/{name}", f"./**/v2/**/{name}", f"./**/data_v2/**/{name}", f"./**/tasks_v2/{name}"]
    for p in pats:
        hits = glob.glob(p, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name}: download the v2/ folder from Hugging Face next to this notebook")

wafers = {s: pd.read_parquet(find(f"wafers_{s}.parquet")) for s in ["train", "val", "test", "test_shift"]}
task1 = {s: pd.read_csv(find(f"task1_{s}.csv")) for s in ["train", "val", "test", "test_shift"]}
meta_val = {m["chuck_id"]: m for m in json.load(open(find("chucks_val.json")))}
{s: (w.chuck_id.nunique(), len(w)) for s, w in wafers.items()}

{'train': (360, 319205),
 'val': (48, 42030),
 'test': (96, 85351),
 'test_shift': (48, 43100)}

## Labels are fault sets

A window's label is `healthy` or the active faults joined by `+` (a fault counts once its severity reaches 0.15).

In [2]:
task1["train"].label.value_counts().head(12)

label
healthy                       5721
contamination                 1339
wear                          1308
arcing                        1167
heater_zone                   1092
sensor_drift                  1083
residual_charge               1002
he_leak                        851
heater_zone+arcing             128
contamination+sensor_drift     117
wear+arcing                    117
contamination+he_leak          111
Name: count, dtype: int64

## Causal features

Each signal is compared with the chuck's own first 100 wafers, recipe by recipe. For each signal: the level now,
the change over the last 30 and 60 wafers, the slope over 90 wafers and the largest 10-wafer step. Plus
consistency checks between signals, a peak-current spike rate (arcing), and the share of wafers on a recipe that
has no reference (`NaN` features there, which the model handles natively). Same code as `escbench/tasks_v2.py`.

In [3]:
import numpy as np
import pandas as pd
FAULTS = ["contamination", "wear", "he_leak", "residual_charge", "heater_zone", "arcing", "sensor_drift"]

REF_WAFERS = 100
SEV_LABEL = 0.15
RUL_FRACS = [0.2, 0.4, 0.6, 0.8]
RATIO = ["i_leak_mean", "he_flow_mean", "he_p_set_mean", "he_p_mean", "i_leak_max"]
DIFF = ["ch_press_mean", "ch_press_max", "t_wafer_mean", "dechuck_time_s", "v_rb_min_frac", "t_zone_spread"]
SERIES = RATIO + DIFF + ["dz_mean", "dz_dev"]
STATS = ["now", "d30", "d60", "slope", "jump"]
FEATURE_NAMES = ([f"{s}_{k}" for s in SERIES for k in STATS] +
                 [f"dz{k}_now" for k in range(1, 5)] +
                 ["p_gap_now", "pset_per_dz", "press_per_flow", "leak_per_flow", "stick_rate", "spike_rate",
                  "frac_unseen_recipe", "wafers_since_install", "wafers_since_clean"])


def normalise(w):
    """Per-wafer series of one chuck relative to its own reference wafers (same recipe)."""
    ref = w.iloc[:REF_WAFERS]
    zk = ["t_z1_mean", "t_z2_mean", "t_z3_mean", "t_z4_mean"]
    cols = RATIO + DIFF + zk
    base = ref.groupby("recipe")[cols].median().reindex(w.recipe.values)
    base.index = w.index
    out = pd.DataFrame(index=w.index)
    for c in RATIO:
        out[c] = w[c] / base[c]
    for c in DIFF:
        out[c] = w[c] - base[c]
    dz = np.column_stack([w[c].values - base[c].values for c in zk])
    for k in range(4):
        out[f"dz{k + 1}"] = dz[:, k]
    out["dz_mean"] = dz.mean(axis=1)
    dev = dz - dz.mean(axis=1, keepdims=True)
    idx = np.nan_to_num(np.abs(dev), nan=-1).argmax(axis=1)
    out["dz_dev"] = dev[np.arange(len(dev)), idx]
    out["unseen"] = base[RATIO[0]].isna().astype(float).values
    # spike flag: peak leakage well above this chuck's reference peak for the recipe
    out["spike"] = (out["i_leak_max"] > 1.25).astype(float)
    out["stick"] = (w["dechuck_time_s"].values > (base["dechuck_time_s"].values + 2.0)).astype(float)
    return out


def _stats(v):
    v = v[~np.isnan(v)]
    n = len(v)
    if n == 0:
        return [np.nan] * 5
    now = np.median(v[-30:])
    p1 = np.median(v[-60:-30]) if n > 40 else now
    p2 = np.median(v[-90:-60]) if n > 70 else p1
    if n > 10:
        x = np.arange(n) - (n - 1) / 2
        slope = float((x * (v - v.mean())).sum() / (x * x).sum() * 30)
    else:
        slope = 0.0
    if n >= 20:
        med = np.array([np.median(v[i:i + 10]) for i in range(0, n - 9, 5)])
        jump = float(np.max(np.abs(np.diff(med)))) if len(med) > 1 else 0.0
    else:
        jump = 0.0
    return [float(now), float(now - p1), float(p1 - p2), slope, jump]


def window_features(w, norm, end):
    lo = max(REF_WAFERS, end + 1 - 90)
    seg = norm.iloc[lo:end + 1]
    cur = norm.iloc[max(REF_WAFERS, end + 1 - 30):end + 1]
    f = []
    for s in SERIES:
        f += _stats(seg[s].values)
    g = dict(zip([f"{s}_{k}" for s in SERIES for k in STATS], f))
    f += [float(np.nanmedian(cur[f"dz{k}"].values)) if cur[f"dz{k}"].notna().any() else np.nan for k in range(1, 5)]
    pset = g["he_p_set_mean_now"] - 1
    f += [g["he_p_mean_now"] - g["he_p_set_mean_now"], pset / (abs(g["dz_mean_now"]) + 0.15),
          g["ch_press_mean_now"] / (abs(g["he_flow_mean_now"] - 1) + 0.1),
          (g["i_leak_mean_now"] - 1) / (abs(g["he_flow_mean_now"] - 1) + 0.1),
          float(cur["stick"].mean()), float(cur["spike"].mean()), float(cur["unseen"].mean()),
          float(w.wafers_since_install.iloc[end]), float(w.wafers_since_clean.iloc[end])]
    return f


def feature_matrix(wafers, rows):
    """rows: DataFrame with chuck_id, end_wafer."""
    groups = {c: g.sort_values("wafer_idx").reset_index(drop=True) for c, g in wafers.groupby("chuck_id")}
    norms = {c: normalise(g) for c, g in groups.items()}
    return np.array([window_features(groups[r.chuck_id], norms[r.chuck_id], int(r.end_wafer)) for r in rows.itertuples()],
                    dtype=float)


def to_multihot(labels):
    """'he_leak+heater_zone' -> 0/1 vector over FAULTS; 'healthy' -> all zeros."""
    Y = np.zeros((len(labels), len(FAULTS)), dtype=int)
    for i, l in enumerate(labels):
        if l != "healthy":
            for f in l.split("+"):
                Y[i, FAULTS.index(f)] = 1
    return Y


def from_multihot(Y):
    names = np.array(FAULTS)
    return ["+".join(names[r.astype(bool)]) if r.any() else "healthy" for r in Y]

In [4]:
%%time
X = {s: feature_matrix(wafers[s], task1[s]) for s in task1}
Y = {s: to_multihot(task1[s].label.values) for s in ["train", "val"]}
X["train"].shape

CPU times: user 1min 13s, sys: 474 ms, total: 1min 13s
Wall time: 1min 13s


(14400, 78)

## Train one classifier per fault

In [5]:
models = [HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, class_weight="balanced", random_state=0)
          .fit(X["train"], Y["train"][:, k]) for k in range(len(FAULTS))]
predict = lambda X_: np.column_stack([m.predict_proba(X_)[:, 1] for m in models]) >= 0.5

def score(Y_true, Y_pred):
    f1 = {}
    for k, f in enumerate(FAULTS):
        t, p = Y_true[:, k].astype(bool), Y_pred[:, k].astype(bool)
        tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
        f1[f] = round(float(2 * tp / max(2 * tp + fp + fn, 1)), 3)
    return float(np.mean(list(f1.values()))), f1

P_val = predict(X["val"])
macro, per_fault = score(Y["val"], P_val)
print("validation macro-F1 over 7 faults:", round(macro, 3))
print("exact match (all faults right):", round(((Y["val"] == P_val).all(1)).mean(), 3))
per_fault

validation macro-F1 over 7 faults: 0.902
exact match (all faults right): 0.89


{'contamination': 0.991,
 'wear': 0.972,
 'he_leak': 0.915,
 'residual_charge': 0.995,
 'heater_zone': 0.737,
 'arcing': 0.931,
 'sensor_drift': 0.776}

## The hard part: which sensor is drifting?

`sensor_drift` means a sensor is wrong but the chuck is fine. How often the model catches it depends heavily on
which sensor drifts. (Validation has only 8 such chucks, so this is rough; the README has test numbers.)

In [6]:
v = task1["val"].assign(pred=from_multihot(P_val))
v["true_sd"] = v.label.str.contains("sensor_drift"); v["caught"] = v.pred.str.contains("sensor_drift")
v["sensor"] = v.chuck_id.map(lambda c: meta_val[c]["drift_sensor"] or "-").str.replace(r"t_z\d_c", "zone_temperature", regex=True)
v[v.true_sd].groupby("sensor").agg(chucks=("chuck_id", "nunique"), recall=("caught", "mean")).round(2)

,chucks,recall
sensor,,
he_flow_sccm,2,0.59
he_p_torr,4,0.98
i_leak_ma,1,0.58
zone_temperature,1,0.12


Ideas to try: a model on the raw 1 Hz signals (published for test and test_shift; rebuild train with
`python generate2.py --splits train --raw` from the GitHub repo), physics-based residuals (expected helium
setpoint given zone temperatures), or more training data from the generator (`--per-class 100 --seed 1`).

## Test predictions

Test answers are held back. To get a score, open an issue on GitHub with these files (`chuck_id,end_wafer,label`).

In [7]:
for s in ["test", "test_shift"]:
    out = task1[s][["chuck_id", "end_wafer"]].assign(label=from_multihot(predict(X[s])))
    out.to_csv(f"v2_task1_{s}_submission.csv", index=False)
out.label.value_counts().head()

label
healthy            429
residual_charge     76
contamination       68
wear                65
arcing              65
Name: count, dtype: int64